# BVI-Net on ISIC2018-a — Colab runner
Run cells top to bottom.

In [ ]:
import torch, sys
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("python:", sys.version)

In [ ]:
!pip install mamba-ssm causal-conv1d --no-build-isolation

In [ ]:
!git clone https://github.com/Ghanasree-S/BVI-Net-ISIC.git
%cd BVI-Net-ISIC
!pip install -q -r requirements.txt

In [ ]:
# Optional: try installing the real Mamba backend. Safe to skip/fail --
# fa_vssm.py auto-falls-back to a lightweight approximation either way.
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mamba-ssm', 'causal-conv1d'],
                          capture_output=True, text=True)
print('mamba-ssm install:', 'OK' if result.returncode == 0 else 'FAILED (fallback SSM will be used)')

In [ ]:
!bash data/download_isic.sh
!python data/prepare_isic_a.py

In [ ]:
# Sanity check: model builds and runs on a dummy batch
import torch
from models import BVINet
m = BVINet()
print('params:', sum(p.numel() for p in m.parameters()))
print('output shape:', m(torch.randn(1, 3, 256, 256)).shape)

In [ ]:
!python train.py --data_dir data/isic2018a --epochs 50 --batch_size 64 --lr 0.001

In [ ]:
!python evaluate.py --checkpoint checkpoints/best.pt --data_dir data/isic2018a --visualize

In [ ]:
# Show a few qualitative results inline
import glob
from PIL import Image
for f in sorted(glob.glob('outputs/sample_*.png'))[:4]:
    display(Image.open(f))